# COMP9085 Week 8 Laboratory
## Instance-based learning: k-nearest neighbours on real patients

**Two hours.** This week's model stores the training data and predicts from the most similar
stored cases. You will build k-NN properly on real clinical data and find out, with your own
numbers, which decisions matter: the value of k, the distance metric, how the features are
represented and scaled, and how neighbours are weighted.

**The data.** 303 patients referred to the Cleveland Clinic for coronary angiography
(1981–1984): the **UCI Heart Disease** dataset, Cleveland subset (Janosi, Steinbrunn, Pfisterer
and Detrano, 1988; CC BY 4.0). Each patient has 13 features from routine measurements and
non-invasive tests; the label is whether the angiogram showed significant coronary artery
disease. The frozen copy is in `data/`: `data/README.md` documents every column and
`data/PROVENANCE.json` the source and checksums. **No network is needed.**

**The question.** For a new patient, which past patients are "similar", and can their
diagnoses predict hers? A good answer says *similar in which features, measured how*.

**By the end of this lab you should be able to**
1. explore a mixed clinical dataset and separate continuous features from coded categories;
2. build a k-NN classifier inside a scikit-learn `Pipeline` and explain one prediction from
   its neighbours;
3. show how k, the distance metric, feature representation, scaling and distance weighting
   change the result;
4. select hyperparameters by cross-validation and evaluate once on a held-out test set;
5. judge the strengths, limits and cost of k-NN, and when another model is preferable.

**Time plan.** Tasks 1–2 about 20 min · Tasks 3–5 about 40 min · Tasks 6–8 about 35 min ·
Tasks 9–10 about 25 min. **Tasks 1–10 are the core lab.** Three optional extensions follow the
checkpoint after Task 10.

**How the cells work.** Cells marked *provided* run as they are: read them and run them.
Cells containing `# your code here` are yours. Some tasks ask you to **predict** first: write
the prediction down before running anything. Markdown cells marked *Your answer here* are
for written answers, using your own numbers.

**Rules for the whole lab.** The data are split once, in a provided cell. Every choice is made
with 10-fold cross-validation on the training patients. The test set is used **once**, in
Task 9. Anything that learns from data (imputer, scaler, encoder) sits inside a `Pipeline`.

**The Week 8 page is your interactive companion**
(https://hafli.github.io/comp9085-ml-biology/weeks/week08.html). Its section 10 maps each
task to the exercise that shows the same idea.

**Practice, not assessment.** This lab is not part of Assignment 1 and uses different data.

**Deliverable.** This notebook, run top to bottom (**Kernel → Restart & Run All**), with every
question answered in the markdown cells.

### Setup (provided)

Run this cell first.

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, roc_auc_score)
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     cross_validate, GridSearchCV)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

import sklearn
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
CV = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
DATA = Path("data")
pd.set_option("display.precision", 3)

## Task 1. The biological problem and the data (about 15 minutes)

Coronary artery disease narrows the arteries that supply the heart muscle. The reference
test, **angiography**, is invasive. The question behind this dataset is whether routine
measurements and non-invasive tests can indicate who has the disease. The features:

| Column | Meaning | Type |
|---|---|---|
| `age` | age (years) | continuous |
| `trestbps` | resting blood pressure (mm Hg) | continuous |
| `chol` | serum cholesterol (mg/dl) | continuous |
| `thalach` | maximum heart rate reached in the exercise test (beats/min) | continuous |
| `oldpeak` | ST-segment depression on exercise relative to rest (mm) | continuous |
| `sex` | 1 = male, 0 = female | binary |
| `cp` | chest-pain type: 1 typical angina, 2 atypical, 3 non-anginal, 4 asymptomatic | categorical (coded 1–4) |
| `fbs` | fasting blood sugar > 120 mg/dl | binary |
| `restecg` | resting ECG: 0 normal, 1 ST-T abnormality, 2 left-ventricular hypertrophy | categorical |
| `exang` | exercise-induced angina | binary |
| `slope` | slope of the peak-exercise ST segment: 1 up, 2 flat, 3 down | categorical |
| `ca` | number of major vessels (0–3) coloured by fluoroscopy | categorical (count) |
| `thal` | thallium scan: 3 normal, 6 fixed defect, 7 reversible defect | categorical (coded) |
| `num` | angiographic diagnosis, 0 (< 50% narrowing) to 4 | **label** |

We predict `disease = (num > 0)`.

**1a (provided).** Load the data.

In [ ]:
df = pd.read_csv(DATA / "heart_disease_cleveland.csv")
df["disease"] = (df["num"] > 0).astype(int)
NUMERIC = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
print(df.shape)
df.head()

**1b.** Inspect the data. Show (i) the data types, (ii) the number of missing values in
each column, (iii) `describe()` for the five continuous features, and (iv) the counts of
`disease` and of the original `num`.

In [ ]:
# your code here


**1c.** Plot a histogram of each continuous feature (one figure, five panels). Look at
the x-axis ranges.

In [ ]:
# your code here


**Questions 1.** (i) Which continuous feature has the largest numerical spread, and which
the smallest? By roughly what factor do their standard deviations differ? (ii) `cp` and `thal`
are stored as integers. Are they measurements? What would `|cp₁ − cp₂| = 3` mean? (iii) How
many values are missing, in which columns, and what will you do about them?

*Your answer here.*

## Task 2. Split, and a baseline (about 10 minutes)

**2a (provided).** One stratified split, 75% training and 25% test. The test set is now put
away until Task 9.

In [ ]:
X = df[NUMERIC + CATEGORICAL]
y = df["disease"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
print("training patients:", len(X_train), "| test patients:", len(X_test))
print("disease share: train %.3f, test %.3f" % (y_train.mean(), y_test.mean()))

**2b.** Estimate the accuracy of a majority-class baseline (`DummyClassifier`) with 10-fold
cross-validation on the training data (`cv=CV`). Store the mean in `baseline_cv`.

In [ ]:
baseline_cv = None
# your code here


**Question 2.** What does the baseline predict for every patient, and why must any model
you report beat it? Why stratify the split?

*Your answer here.*

## Task 3. Build k-NN properly, and read one prediction (about 15 minutes)

k-NN needs every feature to be a number on a sensible scale. Build a preprocessing step that
(i) imputes and standardises the continuous features, and (ii) imputes and **one-hot encodes**
the categorical ones; then a `KNeighborsClassifier(n_neighbors=5)`.

**3a.** Complete `my_knn` and estimate its 10-fold CV accuracy on the training data.

In [ ]:
my_preprocess = ColumnTransformer([
    # ("num", <pipeline: SimpleImputer(strategy="median") then StandardScaler()>, NUMERIC),
    # ("cat", <pipeline: SimpleImputer(strategy="most_frequent") then OneHotEncoder(...)>, CATEGORICAL),
], sparse_threshold=0)
my_knn = None   # Pipeline([("prep", my_preprocess), ("knn", KNeighborsClassifier(5))])
# your code here


**3b (provided).** The same pipeline as a function, so later tasks do not depend on your
cell. Compare its output with yours: they should agree.

In [ ]:
def make_preprocess(scale="standard", encode="onehot"):
    """Continuous: median imputation, then StandardScaler / MinMaxScaler / nothing.
    Categorical: most-frequent imputation, then one-hot ('onehot') or the raw codes ('codes')."""
    scaler = {"standard": [StandardScaler()], "minmax": [MinMaxScaler()], None: []}[scale]
    num = make_pipeline(SimpleImputer(strategy="median"), *scaler)
    if encode == "onehot":
        cat = make_pipeline(SimpleImputer(strategy="most_frequent"),
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    else:
        cat = SimpleImputer(strategy="most_frequent")
    return ColumnTransformer([("num", num, NUMERIC), ("cat", cat, CATEGORICAL)], sparse_threshold=0)


def make_knn(k=5, metric="euclidean", weights="uniform", scale="standard", encode="onehot"):
    return Pipeline([("prep", make_preprocess(scale, encode)),
                     ("knn", KNeighborsClassifier(n_neighbors=k, metric=metric, weights=weights))])


def cv_acc(model, X_=None):
    s = cross_val_score(model, X_train if X_ is None else X_, y_train, cv=CV)
    return s.mean(), s.std()

print("make_knn(5): CV accuracy %.3f (SD %.3f)" % cv_acc(make_knn(5)))

**3c.** Fit `make_knn(5)` on the training data. For the **first patient in `X_train`**,
use `kneighbors()` on the fitted k-NN step (applied to the *transformed* row) to list the 5
nearest training patients, their distances and their labels. Reproduce `predict_proba`
for that patient by counting. *Hint:* `model.named_steps["prep"].transform(...)` and
`model.named_steps["knn"].kneighbors(...)`. What do you notice about the nearest distance?

In [ ]:
# your code here


**Question 3.** Explain in two or three sentences how the five neighbours produce this
prediction. Why is the nearest distance 0 here, and what does that tell you about evaluating
k-NN on its own training data?

*Your answer here.*

## Task 4. How does k change the model? (about 15 minutes)

**Predict first:** at k = 1, what will training accuracy be? Will cross-validated accuracy be
higher or lower than at k = 15?

**4a.** For every odd k from 1 to 51, use `cross_validate(make_knn(k), X_train, y_train, cv=CV,
return_train_score=True)` to record the mean training and CV accuracy. Store a DataFrame
`curve` indexed by k with columns `train` and `cv`, and plot both against k, with the baseline
as a horizontal line.

In [ ]:
K_VALUES = list(range(1, 52, 2))
rows = []
for k in K_VALUES:
    # your code here
    pass
curve = pd.DataFrame(rows)


**Questions 4.** (i) Describe the two curves. Where is the model overfitting, and does it
underfit anywhere in this range? (ii) What would happen at k = 227? (iii) The √n rule suggests
k ≈ 15. Is that a good choice here? (iv) How large is the fold-to-fold standard deviation
compared with the differences between k = 5, 15 and 33?

*Your answer here.*

## Task 5. Distance metrics and feature representation (about 15 minutes)

**5a.** At k = 5 and k = 15, compare `metric="euclidean"` with `metric="manhattan"` (scaled,
one-hot). Report CV mean and SD in a small table.

In [ ]:
# your code here


**5b.** Representation of the categorical features. At k = 5 compare (i) one-hot encoding
with (ii) the raw integer codes (`encode="codes"`), both standardised.

In [ ]:
# your code here


**5c.** Hamming distance is natural for categorical features. Build a k-NN (k = 15) that
uses **only the eight categorical features**, one-hot encoded, with `metric="hamming"`, and
compare it with Euclidean distance on the same one-hot features.

In [ ]:
# your code here


**Questions 5.** (i) Is one metric clearly better? Should you conclude Manhattan "is better
for medical data"? (ii) What changed between one-hot and raw codes, and why can raw codes be
harmful even when the score barely moves? (iii) On one-hot (0/1) vectors, how are Hamming and
Euclidean distance related, and does that explain your 5c result?

*Your answer here.*

## Task 6. Feature scaling (about 15 minutes)

**Predict first:** without scaling, which feature will decide who the neighbours are?

**6a.** At k = 5, compare `scale=None`, `scale="minmax"` and `scale="standard"`.

In [ ]:
# your code here


**6b.** Show *why*. For the five continuous features (training rows, median-imputed but
unscaled), compute each feature's share of the expected squared distance between two random
patients. *Hint:* for two independent random rows, E[(xᵢ − x'ᵢ)²] = 2·Var(xᵢ).

In [ ]:
# your code here


**6c (provided).** Why the scaler belongs inside the Pipeline. The leaky version below fits
the scaler on *all* the training rows before cross-validation, so each validation fold has
already influenced the scaling. Here the effect is small; with feature selection or target
encoding it can be large (Weeks 3 and 5).

In [ ]:
leaky_X = X_train.copy()
leaky_X[NUMERIC] = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(leaky_X[NUMERIC]))
print("scaler fitted outside CV : %.3f" % cv_acc(make_knn(5, scale=None), leaky_X)[0])
print("scaler inside the Pipeline: %.3f" % cv_acc(make_knn(5))[0])

**Questions 6.** (i) Which feature dominates the unscaled distance, with what share, and
which feature is almost ignored? (ii) Explain the accuracy difference in one sentence.
(iii) Why must the scaler be fitted on training data only, even though it never sees the
label? (iv) Min-max or standardisation: which is more sensitive to a single extreme patient?

*Your answer here.*

## Task 7. Model selection by cross-validation (about 15 minutes)

**7a.** Use `GridSearchCV(make_knn(), param_grid, cv=CV)` to search odd k from 1 to 51,
`metric` in {euclidean, manhattan} and `weights` in {uniform, distance}. (Parameter names
inside a Pipeline are `"knn__n_neighbors"`, etc.) Fit on the training data only. Store it as
`search`; print the best parameters and CV score, and the ten best rows of `cv_results_`
with their standard deviations.

In [ ]:
search = None
# your code here


**Questions 7.** (i) Which configuration was chosen? (ii) Look at the ten best rows: are
they meaningfully different? What would you report, and how would you pick between nearly
equal settings (*hint:* the one-standard-error rule)? (iii) Why is the test set not used here,
even though it would be easy to evaluate each configuration on it?

*Your answer here.*

## Task 8. Distance weighting (about 10 minutes)

**8a (provided).** The lecture's worked regression example, checked with scikit-learn: three
stored cases at distances 2, 4 and 6 from the query with values 82, 83 and 78.

In [ ]:
X_toy = np.array([[2.0], [4.0], [6.0]]); y_toy = np.array([82, 83, 78])
for name, w in [("uniform", "uniform"), ("1/d", "distance"), ("1/d^2", lambda d: 1 / d**2)]:
    r = KNeighborsRegressor(n_neighbors=3, weights=w).fit(X_toy, y_toy)
    print("%-8s prediction at x = 0: %.3f" % (name, r.predict([[0.0]])[0]))

**8b.** On the heart data at k = 15 (scaled, Euclidean), compare weights `1/d^p` for p = 0
(uniform), 0.5, 1, 2 and 4, using a callable: `weights=lambda d, p=p: 1 / d**p`.
(scikit-learn passes the array of neighbour distances to the callable.) Report CV accuracy.
Then fit the p = 0 and p = 2 models on all training patients except the first five
(`X_train.iloc[5:]`), and compare their `predict_proba` for those five (`X_train.iloc[:5]`).
The test set stays untouched.

In [ ]:
# your code here


**Questions 8.** (i) Explain the three toy predictions. Which one did the lecture compute?
(ii) What does increasing p do, and what does the model become as p grows very large?
(iii) What happens if a query is at distance 0 from a stored patient, and how does
scikit-learn's `weights="distance"` handle it? Why did 8b guard with `np.maximum(d, 1e-12)`?
(iv) Did weighting change the decisions or mostly the probabilities?

*Your answer here.*

## Task 9. Final evaluation, once (about 15 minutes)

**9a.** Take `search.best_estimator_` (already refitted on all the training data by
`GridSearchCV`), predict the test set **once**, and report accuracy, balanced accuracy, ROC AUC
(from `predict_proba`), the classification report and the confusion matrix.

In [ ]:
# your code here


**9b (provided).** Class balance. The binary label is nearly balanced, but the original
severity label `num` is not. A k-NN asked to predict severity votes among neighbours that are
mostly class 0.

In [ ]:
print(df["num"].value_counts(normalize=True).sort_index().round(3))
sev = cross_val_score(make_knn(15), X_train, df.loc[X_train.index, "num"], cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                      scoring="balanced_accuracy")
print("severity (5 classes), k = 15: balanced accuracy %.3f (chance 0.200)" % sev.mean())

**Questions 9.** (i) Interpret the confusion matrix clinically: what do the false
negatives and false positives mean for a patient? (ii) The test accuracy differs from the CV
estimate. Which number is the better estimate of future performance, and why should you *not*
now change k? (iii) Why is balanced accuracy the better summary for the severity task, and
what happens to rare severe classes under a majority vote? (iv) What population does this
result apply to?

*Your answer here.*

## Task 10. Critical reflection (about 10 minutes)

**10a (provided).** Irrelevant features. Add columns of pure noise and measure the damage
(k = 15). The noise is generated once with a fixed seed.

In [ ]:
rng = np.random.default_rng(SEED)
for n_noise in [0, 20, 50, 100]:
    noise_cols = [f"noise_{j}" for j in range(n_noise)]
    Xn = pd.concat([X_train, pd.DataFrame(rng.normal(size=(len(X_train), n_noise)),
                                          index=X_train.index, columns=noise_cols)], axis=1)
    pre = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
                             ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                                     OneHotEncoder(handle_unknown="ignore", sparse_output=False)), CATEGORICAL),
                             ("noise", "passthrough", noise_cols)], sparse_threshold=0)
    acc = cross_val_score(Pipeline([("prep", pre), ("knn", KNeighborsClassifier(15))]), Xn, y_train, cv=CV).mean()
    print("%3d noise features: CV accuracy %.3f" % (n_noise, acc))

**10b (provided).** Cost. k-NN does its work at prediction time. Compare fitting and
predicting times with logistic regression on a larger, artificially replicated training set.

In [ ]:
big_X = pd.concat([X_train] * 200, ignore_index=True); big_y = pd.concat([y_train] * 200, ignore_index=True)
for name, mdl in [("k-NN (k = 15)", make_knn(15)),
                  ("logistic regression", Pipeline([("prep", make_preprocess()), ("lr", LogisticRegression(max_iter=2000))]))]:
    t0 = time.perf_counter(); mdl.fit(big_X, big_y); t1 = time.perf_counter()
    mdl.predict(X_test); t2 = time.perf_counter()
    print("%-20s fit %.3f s | predict 76 patients %.3f s | training rows stored: %s"
          % (name, t1 - t0, t2 - t1, f"{len(big_X):,}" if "k-NN" in name else "none (coefficients only)"))

**Questions 10.** Answer briefly, using your numbers where you can.
1. Give two strengths of k-NN for this problem.
2. Explain the effect of irrelevant features seen in 10a, and why a decision tree would suffer less.
3. Describe the computational cost of k-NN in training and prediction, and its memory cost.
4. Name a situation in biology or medicine where you would prefer another model to k-NN, and say which model and why.
5. Two patients are nearest neighbours in this feature space. Does that make them biologically or clinically equivalent? Give one reason why not.

*Your answer here.*

---
## Checkpoint: end of the core lab

If you have answered Questions 1–10, you have completed the lab. The extensions are optional.

## Extension A. k-NN regression: predicting maximum heart rate from age

Maximum heart rate falls with age; the textbook rule of thumb is **220 − age**. Using all 303
patients with 10-fold `KFold(10, shuffle=True, random_state=SEED)` cross-validation and the mean
absolute error, compare: `KNeighborsRegressor` with k in {1, 5, 15, 41}, uniform and distance
weights; `LinearRegression`; and the 220 − age rule. Plot the k = 15 prediction curve over the
scatter.

In [ ]:
# your code here


*Your answer here.*

## Extension B. k-NN from scratch

Write `knn_predict(X_tr, y_tr, X_q, k)` in NumPy (Euclidean distance, majority vote, ties
broken towards the smaller label) and check that it agrees with scikit-learn on the
standardised, one-hot test data at k = 5.

In [ ]:
# your code here


*Your answer here.*

## Extension C. A heterogeneous distance

Write a distance for the **raw** (imputed, unscaled, not one-hot) records: for each continuous
feature |difference| / (training range), for each categorical feature 0 if equal and 1 if
not, summed (a Gower-style distance). Use it with `KNeighborsClassifier(15, metric=your_function,
algorithm="brute")` and compare with the one-hot Euclidean pipeline.

In [ ]:
# your code here


*Your answer here.*

## Final reflection

In three or four sentences: what did this lab teach you about the word **"similar"**? Which
single decision (k, metric, representation, scaling, weighting) had the largest effect on these
data, and how do you know?

*Your answer here.*